---
# PROYECTO: EduAI Risk Docente - SPRINT 1: Datos y Dashboard Exploratorio
# EQUIPO: 5
# CURSO: Administración del Desarrollo de Software
# OBTENIBLE:(US03) Como docente, quiero un modelo predictivo sencillo que estime si un estudiante podría aprobar o reprobar.

---

In [6]:
# importar librerias , conectar a google drive y subir data limpio
import numpy as np
import pandas as pd
from google.colab import drive
import matplotlib.pyplot as plt

drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    make_scorer
)

In [10]:
ruta = '/content/drive/MyDrive/MNA/Administración desarrollo de software/student-mat-clean.csv'

dfClean = pd.read_csv(ruta,sep=',')

dfClean.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,goout,Dalc,Walc,health,absences,G1,G2,G3,aprueba,resultado
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,1,1,3,6,5,6,6,0,Reprueba
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,1,1,3,4,5,5,6,0,Reprueba
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,2,2,3,3,10,7,8,10,1,Aprueba
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,1,1,5,2,15,14,15,1,Aprueba
4,GP,F,16,U,GT3,T,3,3,other,other,...,2,1,2,5,4,6,10,10,1,Aprueba


In [20]:
# Preparación del conjunto de entrenamiento

#Selección de las features

features = ["G1", "G2", "absences", "studytime", "failures"]
x = dfClean[features]

# selección de la variable objetivo (aprueba o reprueba )
y = dfClean["aprueba"]  # aprueba = 0 , reprueba= 1


print("Distribución de clases:")
print(y.value_counts())

Distribución de clases:
aprueba
1    265
0    130
Name: count, dtype: int64


In [24]:
# división del conjunto de entrnamiento y validación
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size =0.20,
    random_state= 42,
    stratify= y
)

In [27]:
# pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ])

In [28]:
# 5. Prueba de hiperparámetros


param_grid = {
    "modelo__C": [0.01, 0.1, 1, 10, 100],
    "modelo__penalty": ["l1", "l2"],
    "modelo__solver": ["liblinear"]
}

scoring = {
    "Accuracy": "accuracy",
    "F1-score": make_scorer(f1_score, zero_division=0),
    "Precision": make_scorer(precision_score, zero_division=0),
    "Recall": make_scorer(recall_score, zero_division=0)
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit="F1-score",
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print("\nMejores hiperparámetros:")
print(grid_search.best_params_)

print("\nMejor F1-score en validación cruzada:")
print(grid_search.best_score_)


Mejores hiperparámetros:
{'modelo__C': 10, 'modelo__penalty': 'l1', 'modelo__solver': 'liblinear'}

Mejor F1-score en validación cruzada:
0.9390213652313701


In [29]:
# 6. Evaluar modelo final


best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, zero_division=0)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print("\nMétricas del modelo en datos de prueba:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nReporte de clasificación:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Aprueba", "Reprueba"],
    zero_division=0
))


Métricas del modelo en datos de prueba:
Accuracy:  0.8734
F1-score:  0.8980
Precision: 0.9778
Recall:    0.8302

Matriz de confusión:
[[25  1]
 [ 9 44]]

Reporte de clasificación:
              precision    recall  f1-score   support

     Aprueba       0.74      0.96      0.83        26
    Reprueba       0.98      0.83      0.90        53

    accuracy                           0.87        79
   macro avg       0.86      0.90      0.87        79
weighted avg       0.90      0.87      0.88        79



In [30]:
# ==============================
# 7. Ver resultados de hiperparámetros


resultados_grid = pd.DataFrame(grid_search.cv_results_)

columnas_resultado = [
    "param_modelo__C",
    "param_modelo__penalty",
    "param_modelo__solver",
    "mean_test_Accuracy",
    "mean_test_F1-score",
    "mean_test_Precision",
    "mean_test_Recall"
]

print("\nResultados de la búsqueda de hiperparámetros:")
print(
    resultados_grid[columnas_resultado]
    .sort_values(by="mean_test_F1-score", ascending=False)
)


Resultados de la búsqueda de hiperparámetros:
   param_modelo__C param_modelo__penalty param_modelo__solver  \
9           100.00                    l2            liblinear   
8           100.00                    l1            liblinear   
6            10.00                    l1            liblinear   
4             1.00                    l1            liblinear   
7            10.00                    l2            liblinear   
5             1.00                    l2            liblinear   
3             0.10                    l2            liblinear   
2             0.10                    l1            liblinear   
1             0.01                    l2            liblinear   
0             0.01                    l1            liblinear   

   mean_test_Accuracy  mean_test_F1-score  mean_test_Precision  \
9            0.920784            0.939021             0.965379   
8            0.920784            0.939021             0.965379   
6            0.920784            0.9390